# XGBoost

Same features, same time split, same discipline — a second gradient booster to see whether the score is model-limited or data-limited. If XGBoost lands on top of LightGBM, the features are the ceiling, and no amount of model swapping moves it.

**Same features, same split, same discipline.** The only variable changing in this notebook is the library. Everything else — feature set, time split, categorical handling — is held identical to notebook 07, which is what makes the comparison meaningful rather than decorative.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from src.data import INTERIM
from src.models import time_split

CUTOFF, CATS = "2015-07-01", ["deviceID", "paymentMethod"]
DROP = ["orderID", "orderDate", "customerID", "any_return"]

df = pd.read_parquet(INTERIM / "orders.parquet")
for block in ["basket", "customer", "article"]:
    df = df.merge(pd.read_parquet(INTERIM / f"{block}.parquet"), on="orderID", how="left")
FEATS = [c for c in df.columns if c not in DROP]
X = df[FEATS].copy()
for c in CATS:
    X[c] = X[c].astype("category")
y = df.any_return
tr, va = time_split(df, "orderDate", CUTOFF)
xgb.__version__, X.shape

('3.3.0', (738698, 36))

## Fit

**Matched configuration.** `tree_method="hist"` is the histogram algorithm that makes LightGBM fast, and `enable_categorical=True` gives XGBoost native categorical handling, so neither model gets an unfair encoding advantage.

In [2]:
model = xgb.XGBClassifier(n_estimators=800, learning_rate=0.03, max_depth=6,
                          subsample=0.8, colsample_bytree=0.7, reg_lambda=2.0,
                          min_child_weight=10, tree_method="hist", enable_categorical=True,
                          n_jobs=-1, eval_metric="logloss")
model.fit(X[tr], y[tr])
pred_xgb = model.predict_proba(X[va])[:, 1]

def ece_of(pred, actual, bins=10):
    d = pd.DataFrame({"p": pred, "a": actual})
    d["b"] = pd.qcut(d.p, bins, labels=False, duplicates="drop")
    g = d.groupby("b").agg(n=("p", "size"), p=("p", "mean"), a=("a", "mean"))
    return float((g.n / g.n.sum() * (g.p - g.a).abs()).sum())

res_xgb = {"auc": roc_auc_score(y[va], pred_xgb), "logloss": log_loss(y[va], pred_xgb),
           "brier": brier_score_loss(y[va], pred_xgb), "ece": ece_of(pred_xgb, y[va].values)}
print(f"XGBoost   AUC {res_xgb['auc']:.4f}   logloss {res_xgb['logloss']:.4f}   ECE {res_xgb['ece']:.4f}")

XGBoost   AUC 0.8528   logloss 0.4524   ECE 0.0050


## Against LightGBM

**The result is the finding.** XGBoost wins by 0.0002 AUC — noise. The score is *data-limited*, not model-limited: the features define the ceiling, and swapping algorithms will not move it. Anyone promising a large lift from a different library here is selling something.

In [3]:
lgb_res = json.loads((INTERIM / "model.json").read_text())
try:
    lr_res = json.loads((INTERIM / "regression.json").read_text())
except FileNotFoundError:
    lr_res = None

rows = [{"model": "logistic regression", **{k: lr_res[k] for k in ["auc", "logloss"]}} ] if lr_res else []
rows += [{"model": "LightGBM", "auc": lgb_res["auc"], "logloss": lgb_res["logloss"], "ece": lgb_res["ece"]},
         {"model": "XGBoost", "auc": res_xgb["auc"], "logloss": res_xgb["logloss"], "ece": res_xgb["ece"]}]
cmp = pd.DataFrame(rows)
cmp["auc_gap_vs_lgb"] = cmp.auc - lgb_res["auc"]
(INTERIM / "xgboost.json").write_text(json.dumps(res_xgb, indent=2))
cmp.round(4)

,model,auc,logloss,ece,auc_gap_vs_lgb
0,logistic regression,0.8348,0.4819,NaN,-0.0178
1,LightGBM,0.8526,0.4527,0.0047,0.0000
2,XGBoost,0.8528,0.4524,0.0050,0.0002


## Agreement

**How alike are they really?** Predictions correlate 0.9978 and blending buys another 0.0001. Two independently implemented boosters converging this tightly is strong evidence the ceiling is the information available, not the fitting procedure.

In [4]:
pred_lgb = pd.read_parquet(INTERIM / "predictions.parquet").pred.values
print(f"correlation between the two models' predictions: {np.corrcoef(pred_lgb, pred_xgb)[0,1]:.4f}")
print(f"mean absolute difference: {np.abs(pred_lgb - pred_xgb).mean():.4f}")
print()
print("Blend (simple average):")
blend = (pred_lgb + pred_xgb) / 2
print(f"  AUC {roc_auc_score(y[va], blend):.4f}   logloss {log_loss(y[va], blend):.4f}")

correlation between the two models' predictions: 0.9978
mean absolute difference: 0.0120

Blend (simple average):
  AUC 0.8529   logloss 0.4523
